# Activation Checkpointing Tutorial

## Overview

Trade compute for memory by recomputing activations during backward pass.

### Memory Savings
$$M_{\text{saved}} = M_{\text{activations}} \times (1 - 1/\sqrt{L})$$

Where L = number of layers

In [ ]:
import torch
from torch.utils.checkpoint import checkpoint

class CheckpointedTransformerBlock(torch.nn.Module):
    """Transformer block with activation checkpointing."""
    
    def __init__(self, d_model, nhead):
        super().__init__()
        self.attn = torch.nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.ffn = torch.nn.Sequential(
            torch.nn.Linear(d_model, 4 * d_model),
            torch.nn.GELU(),
            torch.nn.Linear(4 * d_model, d_model),
        )
        self.norm1 = torch.nn.LayerNorm(d_model)
        self.norm2 = torch.nn.LayerNorm(d_model)
    
    def forward(self, x, use_checkpoint=True):
        if use_checkpoint and self.training:
            # Checkpoint attention
            x = x + checkpoint(self._attn_block, x, use_reentrant=False)
            # Checkpoint FFN
            x = x + checkpoint(self._ffn_block, x, use_reentrant=False)
        else:
            x = x + self._attn_block(x)
            x = x + self._ffn_block(x)
        return x
    
    def _attn_block(self, x):
        x = self.norm1(x)
        x, _ = self.attn(x, x, x)
        return x
    
    def _ffn_block(self, x):
        return self.ffn(self.norm2(x))

## Trade-offs

| Aspect | Without Checkpoint | With Checkpoint |
|--------|-------------------|----------------|
| Memory | O(L) | O(sqrt(L)) |
| Compute | 1x | ~1.3x |
| Use when | Memory available | Memory constrained |